# M3 Smart Triage — AI-Powered Priority Classification & Audit

## Purpose
ML-based automatic priority classification with LLM audit and discrepancy detection.

## Priority Definition Methodology

### Contractual Definitions (SLB SOW)

**P1 - Critical** (4h resolution, 24×7):
- Complete failure, NO workaround
- Affects MOST SLB end users or entire division
- Critical business period (month-end, year-end)
- Financial impact >$500K

**P2 - High** (8h resolution, 24×7):
- Major issues, SIGNIFICANT user base impact
- Senior management involvement
- Time-sensitive business processes
- Financial impact $50K-$500K

**P3 - Moderate** (72 business hours, 24×5):
- Limited users, recurring basis
- Partial functionality loss
- Work is impeded but not blocked

**P4 - Standard** (120 business hours, 24×5):
- Isolated, non-recurring
- Workaround available
- Minimal business impact

## Priority Audit Methodology

### Dual-Path Approach

**Path 1: LLM-Based (Primary)**
```
Input: Incident number, description, service, current priority
Model: GPT-4 via Capgemini OpenAI endpoint
System Prompt: 20 years ITSM expertise, conservative P1 classification
Output: {verdict, suggested_priority, confidence [0-1], reasoning}
Confidence Threshold: >70% to recommend change
```

**Path 2: Rule-Based Fallback**
```
Triggers for escalation:
- P1 Keywords: 'outage', 'production down', 'all users', '$500K'
- P2 Keywords: 'major', 'significant users', 'senior management'
- P4 Keywords: 'isolated', 'workaround', 'minimal impact'

Used when: LLM unavailable or latency >5s
```

### Audit Results Processing
```
CORRECT = Current priority matches audit recommendation
RECLASSIFY = Audit suggests different priority

Statistics:
- % correctly_classified (should be >95%)
- % should_escalate (over-classified as lower priority)
- % should_downgrade (over-prioritized)
```

## ML Classification Model (Future)

**Features**:
- TF-IDF vectorization of: short_description, resolution_notes
- Numerical: reassignment_count, reopen_count, category_encoded
- Categorical: service_offering, assignment_group encoded

**Model**: Random Forest (100 trees, max_depth=10)
**Training Data**: Historical incidents with made_sla as proxy for correct priority
**Accuracy Target**: >88% on holdout test set
**Cross-validation**: 5-fold stratified

## Key Metrics
- **Precision per priority**: TP / (TP + FP)
- **Recall per priority**: TP / (TP + FN)
- **F1-score**: Harmonic mean of precision/recall
- **Confusion matrix**: P1-P4 misclassification patterns


In [ ]:
import requests
import pandas as pd
import numpy as np

# Get priority definitions
API_BASE = 'http://127.0.0.1:8002/api'

response = requests.get(f'{API_BASE}/triage/priority-definitions')
defs = response.json()

for p, definition in defs.items():
    print(f"\nP{p}: {definition['label']}")
    print(f"  Response: {definition['response_sla']}")
    print(f"  Resolution: {definition['resolution_sla']}")
    print(f"  Criteria: {definition['full_criteria'][:80]}...")

In [ ]:
# Run priority audit
response = requests.post(f'{API_BASE}/triage/priority-audit?max_incidents=20')
audit = response.json()

print(f"Incidents Audited: {audit.get('total_audited', 0)}")
print(f"Correctly Classified: {audit.get('correctly_classified', 0)} ({audit.get('correctly_classified', 0) / max(1, audit.get('total_audited', 1)) * 100:.1f}%)")
print(f"Should Escalate: {audit.get('should_escalate', 0)}")
print(f"Should Downgrade: {audit.get('should_downgrade', 0)}")